In [ ]:


class ManualRNN(nn.Module):

    def __init__(self, input_size):
        super().__init__()
        self.W_xh = nn.Parameter(torch.randn(input_size, HIDDEN_SIZE) * 0.01)
        self.W_hh = nn.Parameter(torch.randn(HIDDEN_SIZE, HIDDEN_SIZE) * 0.01)
        self.b = nn.Parameter(torch.zeros(HIDDEN_SIZE))
        self.output = nn.Linear(HIDDEN_SIZE, 1)

    def forward(self, x):
        batch_size = x.shape[0]
        h = torch.zeros(batch_size, HIDDEN_SIZE, device=x.device)

        for t in range(x.shape[1]):
            x_t = x[:, t, :]
            h = torch.tanh(x_t @ self.W_xh + h @ self.W_hh + self.b)

        return self.output(h).squeeze(1)


class ManualGRU(nn.Module):

    def __init__(self, input_size):
        super().__init__()

        self.W_x = nn.Parameter(torch.randn(input_size, 3 * HIDDEN_SIZE) * 0.01)
        self.W_h = nn.Parameter(torch.randn(HIDDEN_SIZE, 3 * HIDDEN_SIZE) * 0.01)
        self.b_x = nn.Parameter(torch.zeros(3 * HIDDEN_SIZE))
        self.b_h = nn.Parameter(torch.zeros(3 * HIDDEN_SIZE))

        self.output = nn.Linear(HIDDEN_SIZE, 1)

    def forward(self, x):
        batch_size = x.shape[0]
        h = torch.zeros(batch_size, HIDDEN_SIZE, device=x.device)

        for t in range(x.shape[1]):
            x_t = x[:, t, :]

            x_gates = x_t @ self.W_x + self.b_x
            h_gates = h @ self.W_h + self.b_h

            x_z, x_r, x_n = x_gates.chunk(3, dim=1)
            h_z, h_r, h_n = h_gates.chunk(3, dim=1)

            z = torch.sigmoid(x_z + h_z)
            r = torch.sigmoid(x_r + h_r)
            n = torch.tanh(x_n + r * h_n)

            h = (1 - z) * n + z * h

        return self.output(h).squeeze(1)


class ManualLSTM(nn.Module):


    def __init__(self, input_size):
        super().__init__()

        self.W_x = nn.Parameter(torch.randn(input_size, 4 * HIDDEN_SIZE) * 0.01)
        self.W_h = nn.Parameter(torch.randn(HIDDEN_SIZE, 4 * HIDDEN_SIZE) * 0.01)
        self.b = nn.Parameter(torch.zeros(4 * HIDDEN_SIZE))

    
        with torch.no_grad():
            self.b[HIDDEN_SIZE : 2 * HIDDEN_SIZE] = 1.0

        self.output = nn.Linear(HIDDEN_SIZE, 1)

    def forward(self, x):
        batch_size = x.shape[0]
        h = torch.zeros(batch_size, HIDDEN_SIZE, device=x.device)
        c = torch.zeros(batch_size, HIDDEN_SIZE, device=x.device)

        for t in range(x.shape[1]):
            x_t = x[:, t, :]

            gates = x_t @ self.W_x + h @ self.W_h + self.b
            i, f, g, o = gates.chunk(4, dim=1)

            i = torch.sigmoid(i)
            f = torch.sigmoid(f)
            g = torch.tanh(g)
            o = torch.sigmoid(o)

            c = f * c + i * g
            h = o * torch.tanh(c)

        return self.output(h).squeeze(1)



def train_one_model(model, train_loader, val_loader):
    model = model.to(DEVICE)

    loss_function = nn.MSELoss()
    optimizer = torch.optim.Adam(model.parameters(), lr=LEARNING_RATE)

    best_val_loss = float("inf")
    best_weights = None
    bad_epochs = 0

    train_losses = []
    val_losses = []

    for epoch in range(1, EPOCHS + 1):
        model.train()
        train_loss_sum = 0

        for X_batch, y_batch in train_loader:
            X_batch = X_batch.to(DEVICE)
            y_batch = y_batch.to(DEVICE)

            prediction = model(X_batch)
            loss = loss_function(prediction, y_batch)

            optimizer.zero_grad()
            loss.backward()
            torch.nn.utils.clip_grad_norm_(model.parameters(), max_norm=1.0)
            optimizer.step()

            train_loss_sum += loss.item() * len(y_batch)

        train_loss = train_loss_sum / len(train_loader.dataset)

        # -------- валидацияя --------
        model.eval()
        val_loss_sum = 0

        with torch.no_grad():
            for X_batch, y_batch in val_loader:
                X_batch = X_batch.to(DEVICE)
                y_batch = y_batch.to(DEVICE)

                prediction = model(X_batch)
                loss = loss_function(prediction, y_batch)
                val_loss_sum += loss.item() * len(y_batch)

        val_loss = val_loss_sum / len(val_loader.dataset)

        train_losses.append(train_loss)
        val_losses.append(val_loss)

        print(
            f"Эпоха {epoch:02d}: "
            f"train MSE = {train_loss:.5f}, "
            f"val MSE = {val_loss:.5f}"
        )

        # -------- ранний стоп --------
        if val_loss < best_val_loss:
            best_val_loss = val_loss
            best_weights = model.state_dict().copy()
            bad_epochs = 0
        else:
            bad_epochs += 1

        if bad_epochs >= PATIENCE:
            print("Ранняя остановка")
            break

    if best_weights is not None:
        model.load_state_dict(best_weights)

    return model, train_losses, val_losses







def inverse_target(y_scaled, target_scaler):
    # предсказания из стандартизированного log1p(cnt) обратно в cnt
    y_log = target_scaler.inverse_transform(y_scaled.reshape(-1, 1)).reshape(-1)
    y_cnt = np.expm1(y_log)
    return np.maximum(y_cnt, 0)


def evaluate_model(model, test_loader, target_scaler):
    model.eval()

    predictions = []
    answers = []

    with torch.no_grad():
        for X_batch, y_batch in test_loader:
            X_batch = X_batch.to(DEVICE)
            pred = model(X_batch).cpu().numpy()

            predictions.append(pred)
            answers.append(y_batch.numpy())

    predictions = np.concatenate(predictions)
    answers = np.concatenate(answers)

    y_pred = inverse_target(predictions, target_scaler)
    y_true = inverse_target(answers, target_scaler)

    rmse = np.sqrt(mean_squared_error(y_true, y_pred))
    mae = mean_absolute_error(y_true, y_pred)
    r2 = r2_score(y_true, y_pred)
    mape = np.mean(np.abs((y_true - y_pred) / np.maximum(y_true, 1))) * 100

    return {
        "RMSE": rmse,
        "MAE": mae,
        "MAPE_%": mape,
        "R2": r2,
    }, y_true, y_pred








def plot_losses(all_losses):
    plt.figure(figsize=(10, 6))

    for model_name, losses in all_losses.items():
        plt.plot(losses["val"], label=model_name)

    plt.title("Validation loss")
    plt.xlabel("Эпоха")
    plt.ylabel("MSE")
    plt.legend()
    plt.grid(alpha=0.3)
    plt.tight_layout()
    plt.savefig(RESULTS_DIR / "validation_loss.png", dpi=160)
    plt.close()


def plot_prediction(y_true, y_pred, model_name):
    points = min(240, len(y_true))

    plt.figure(figsize=(12, 5))
    plt.plot(y_true[:points], label="Факт")
    plt.plot(y_pred[:points], label="Прогноз")
    plt.title(f"Лучшая модель: {model_name}")
    plt.xlabel("Час в test выборке")
    plt.ylabel("Количество арендованных велосипедов")
    plt.legend()
    plt.grid(alpha=0.3)
    plt.tight_layout()
    directory =  "model_predictions" + model_name + ".png"
    plt.savefig(RESULTS_DIR / directory, dpi=160)
    plt.close()


def main():
    RESULTS_DIR.mkdir(exist_ok=True)

    print(f"Устройство: {DEVICE}")
    print("data/hour.csv")

    X_train, y_train, X_val, y_val, X_test, y_test, target_scaler = prepare_data()

    train_loader = DataLoader(
        BikeDataset(X_train, y_train),
        batch_size=BATCH_SIZE,
        shuffle=True,
    )
    val_loader = DataLoader(
        BikeDataset(X_val, y_val),
        batch_size=BATCH_SIZE,
        shuffle=False,
    )
    test_loader = DataLoader(
        BikeDataset(X_test, y_test),
        batch_size=BATCH_SIZE,
        shuffle=False,
    )

    input_size = X_train.shape[2]

    print(f"Размерность одного часа после обработки признаков: {input_size}")
    print(f"Train: {len(X_train)} примеров")
    print(f"Validation: {len(X_val)} примеров")
    print(f"Test: {len(X_test)} примеров")

    models = {
        "PyTorch RNN": PyTorchRNN(input_size),
        "PyTorch GRU": PyTorchGRU(input_size),
        "PyTorch LSTM": PyTorchLSTM(input_size),
        "Manual RNN": ManualRNN(input_size),
        "Manual GRU": ManualGRU(input_size),
        "Manual LSTM": ManualLSTM(input_size),
    }

    results = []
    all_losses = {}
    all_predictions = {}

    
    for model_name, model in models.items():
        print("\n" + "=" * 60)
        print(model_name)
        print("=" * 60)

        trained_model, train_losses, val_losses = train_one_model(
            model,
            train_loader,
            val_loader,
        )

        metrics, y_true, y_pred = evaluate_model(
            trained_model,
            test_loader,
            target_scaler,
        )

        metrics["model"] = model_name
        results.append(metrics)

        all_losses[model_name] = {
            "train": train_losses,
            "val": val_losses,
        }
        all_predictions[model_name] = {
            "true": y_true,
            "pred": y_pred,
        }
        plot_prediction(y_true, y_pred, model_name)


        print(
            f"TEST: RMSE = {metrics['RMSE']:.2f}, "
            f"MAE = {metrics['MAE']:.2f}, "
            f"MAPE = {metrics['MAPE_%']:.2f}%, "
            f"R2 = {metrics['R2']:.4f}"
        )

    results_df = pd.DataFrame(results)
    results_df = results_df.sort_values("RMSE").reset_index(drop=True)

    results_df.to_csv(RESULTS_DIR / "metrics.csv", index=False)

    best_model_name = results_df.loc[0, "model"]
    best_true = all_predictions[best_model_name]["true"]
    best_pred = all_predictions[best_model_name]["pred"]

    plot_losses(all_losses)
    plot_prediction(best_true, best_pred, best_model_name)


    print("\n" + "=" * 60)
    print("ИТОГОВОЕ СРАВНЕНИЕ НА TEST")
    print("=" * 60)
    print(results_df.to_string(index=False))

    print("\nЛучшая модель по RMSE:", best_model_name)
    print("\nФайлы сохранены в results/:")
    print("metrics.csv")
    print("validation_loss.png")
    print("best_model_predictions.png")


if __name__ == "__main__":
    main()